# 01 · Data preparation — Ausgrid Solar Home + Open-Meteo weather

**Paper 1 — Multimodal Spatiotemporal Transformer for Urban Energy Forecasting**

This notebook downloads the two **real, public** data sources and builds the aligned,
leakage-safe arrays used by every later notebook:

1. **Ausgrid Solar Home Electricity Data** — *real* 30-minute demand (`GC`) and solar generation
   (`GG`) for Sydney households, **2010-07-01 → 2013-06-30**, per customer and postcode. It is the
   public dataset that co-meters *demand and PV on the same node*, which the joint-forecasting
   research question requires.
2. **Open-Meteo Historical Weather API** (ERA5-backed, keyless) — hourly temperature, humidity,
   shortwave radiation, wind and precipitation for Sydney.

> **Integrity note.** Nothing here is synthetic. If a download fails, the pipeline *raises* — it
> never substitutes noise or simulated data. All paper numbers come from running this on the real files.

**Prerequisite:** download the Ausgrid CSV manually (the site blocks automated fetching) and place
it at `data/raw/` per the README. The data module looks for a `.parquet` cache, then falls back to
the raw CSV.

## 0 · Environment

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/<your-user>/paper1-urban-energy-forecasting.git
    %cd paper1-urban-energy-forecasting
    !pip -q install -r requirements.txt
else:
    %cd ..   # notebook lives in notebooks/; repo root is one up

sys.path.insert(0, os.getcwd())
print('repo root:', os.getcwd())

In [ ]:
import numpy as np, pandas as pd, yaml
from pathlib import Path
from src.utils import load_config, ensure_dir

cfg = load_config('configs/default.yaml')
print(yaml.safe_dump(cfg['data'], sort_keys=False))

## 1 · Load the raw Ausgrid file

The Ausgrid file is wide: one row per (customer, day) with 48 half-hourly columns for each of
`GC` (consumption) and `GG` (gross generation), plus static customer metadata (postcode, customer
class). `load_ausgrid_raw` melts the 48 columns to a long (timestamp, demand, gen) frame and
resamples to hourly; it raises if the file is missing.

In [ ]:
from src.data.dataset import load_ausgrid_raw, WEATHER_COLS

raw_dir = Path(cfg['data']['raw_dir'])
csv_candidates = sorted(raw_dir.glob('*.csv')) if raw_dir.exists() else []
assert csv_candidates, (
    f'No CSV found under {raw_dir}. Download the Ausgrid Solar Home CSV (see README) and place it there.\n'
    'https://www.ausgrid.com.au/Industry/Our-Research/Data-to-share/Solar-home-electricity-data'
)
print('raw files found:', [p.name for p in csv_candidates])

hourly = load_ausgrid_raw(raw_dir)
print('hourly rows:', len(hourly), '| columns:', list(hourly.columns))

### Sanity checks on the real data
Timestamp span must cover 2010-07 → 2013-06; demand strictly positive; generation ≥ 0 with a
clear midday solar peak (the signature of real PV, not noise).

In [ ]:
ts = pd.to_datetime(hourly['timestamp'])
print('span:', ts.min(), '->', ts.max())
print('demand kWh min/median/max: %.4f / %.4f / %.4f' % (
    hourly['demand_kwh'].min(), hourly['demand_kwh'].median(), hourly['demand_kwh'].max()))
print('gen kWh    min/median/max: %.4f / %.4f / %.4f' % (
    hourly['gen_kwh'].min(), hourly['gen_kwh'].median(), hourly['gen_kwh'].max()))

hourly['hour'] = ts.dt.hour
print('\nmean generation by hour (kWh) — expect a midday peak:')
print(hourly.groupby('hour')['gen_kwh'].mean().round(4).to_string())

## 2 · Fetch real weather from Open-Meteo

`fetch_weather` pulls the ERA5 archive for the data window and caches the result. It **raises** on
network failure rather than substituting synthetic weather.

In [ ]:
import inspect
from src.data.dataset import fetch_weather
print('fetch_weather signature:', inspect.signature(fetch_weather))

The data module calls `fetch_weather` internally during `setup()`, so we don't invoke it by hand
here — the next cell runs the whole `AusgridDataModule.setup()` pipeline, which fetches weather,
builds the tensor, the graph and the datasets.

## 3 · Build the aligned tensor + train-only graph

`AusgridDataModule.setup()` performs, in order:
1. **Load** the hourly frame (parquet cache, else raw CSV).
2. **Chronological split** — train `2010-07-01 → 2012-06-30`, test `2012-07-01 → 2013-06-30`.
3. **Fetch weather** (Open-Meteo/ERA5) for the window.
4. **Train-only scaling** — scalers for demand, generation, weather and 16 building features are
   fit **only on the train slice**.
5. **Train-only graph** — postcode adjacency by (≤30 km) ∩ (demand-profile cosine ≥ 0.7), top-`K`.
6. **Leakage-safe subgraph datasets** — each sample's star subgraph windows end at its anchor hour.

In [ ]:
from src.data.dataset import AusgridDataModule

dm = AusgridDataModule(cfg)
dm.setup()   # loads data, fetches weather, fits scalers, builds graph + datasets

print('nodes (co-metered customers):', dm.train_ds.n_nodes)
print('train samples:', len(dm.train_ds), '| test samples:', len(dm.test_ds))
print('K neighbours per node:', dm.train_ds.K)
print('demand scaler mean/std: %.3f / %.3f' % (
    dm.demand_scaler.mean_[0], dm.demand_scaler.scale_[0]))
print('weather columns:', WEATHER_COLS)

## 4 · Verify the leakage invariants on the *real* tensors

Re-run the numpy-only integrity tests, then spot-check one real sample: the input window must end
at the anchor and all four targets must be strictly after it.

In [ ]:
!python tests/test_leakage_and_graph.py

In [ ]:
import torch
s = dm.train_ds[0]
n, a = s['meta'].tolist()
print(f'sample node={n} anchor_hour={a}')
print('input window (L, C):', tuple(s['x_temporal'].shape))
print('root subgraph (K+1, L, C):', tuple(s['sub_temporal'].shape))
print('edge_index:', tuple(s['sub_edge_index'].shape),
      '| weights:', tuple(s['sub_edge_weight'].shape),
      '| bias:', tuple(s['sub_edge_bias'].shape))
print('targets (demand kWh):', dm.demand_scaler.inverse_transform(
    s['y_demand'].numpy().reshape(-1, 1)).ravel().round(3))

assert torch.allclose(s['x_temporal'][-1], torch.from_numpy(dm.train_ds.data[n, a]), atol=1e-5)
print('OK: input window ends exactly at the anchor — no future leakage.')

## 5 · Persist the processed pipeline

Save the fully-set-up data module so notebooks 02–04 load deterministically (same split, same
graph, same scalers every run).

In [ ]:
import pickle
proc = Path(cfg['data']['processed_dir']); ensure_dir(proc)
with open(proc / 'dataset.pkl', 'wb') as f:
    pickle.dump(dm, f)
print('saved ->', proc / 'dataset.pkl')